In [1]:
import os
import numpy as np
import mne
import matplotlib.pyplot as plt
from mne.time_frequency import tfr_multitaper

mne.set_log_level("INFO")

In [3]:
os.chdir(r"C:\Users\Yulia\Desktop\Master thesis") 
folder = "sub-001"
sub = "sub-001"

conditions = ["sivi", "sive", "covi", "cove"]

# окна
baseline = (-3.0, 0.0)
roi = (0.0, 3.5)

# частоты (можно расширить до 80)
freqs = np.arange(2, 41, 1)

# параметры TF
decim = 5
time_bandwidth = 4.0

# куда сохранять
out_dir = os.path.join(folder, "TFR_10plots")
os.makedirs(out_dir, exist_ok=True)

In [14]:
def save_tfr_heatmap(mat_f_t, times, freqs, title, fname):

    # симметричная шкала вокруг 0 (устойчиво к выбросам)
    v = np.percentile(np.abs(mat_f_t), 98)

    plt.figure(figsize=(10, 4))
    plt.imshow(
        mat_f_t,
        aspect="auto",
        origin="lower",
        extent=[times[0], times[-1], freqs[0], freqs[-1]],
        cmap="RdBu_r",          # дивергентная палитра
        vmin=-v,
        vmax=v,
        interpolation="nearest" # без сглаживания
    )

    plt.xlabel("Time (s)")
    plt.ylabel("Frequency (Hz)")
    plt.title(title)
    plt.colorbar(label="Power change (dB)")
    plt.axvline(0, color="k", linewidth=1)

    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, fname), dpi=200)
    plt.close()

In [16]:
raw_maps = {}   # cond -> (freq x time)
db_maps  = {}   # cond -> (freq x time)
roi_times = None

for cond in conditions:
    fname = os.path.join(folder, f"{sub}_{cond}-epo.fif")
    epochs = mne.read_epochs(fname, preload=True).pick("grad")

    # TF по trials
    power = epochs.compute_tfr(
        method="multitaper",
        freqs=freqs,
        n_cycles=freqs / 2.0,
        time_bandwidth=time_bandwidth,
        return_itc=False,
        average=False,   # trial-wise
        decim=decim,
        n_jobs=1
    )

    # усреднение по trials
    power_avg = power.average()  # (n_ch, n_freq, n_time)

    # усреднение по всем градиометрам -> (n_freq, n_time)
    tf_raw = power_avg.data.mean(axis=0)

    # baseline correction: logratio = log10(power / baseline)
    power_db = power.copy()
    power_db.apply_baseline(baseline=baseline, mode="logratio")  # log10 ratio
    power_db_avg = power_db.average()
    tf_db = 10.0 * power_db_avg.data.mean(axis=0)  # умножаем на 10 -> dB

    # режем по времени 0..3.5
    tmask = (power_avg.times >= roi[0]) & (power_avg.times <= roi[1])
    times_roi = power_avg.times[tmask]
    tf_raw_roi = tf_raw[:, tmask]
    tf_db_roi  = tf_db[:, tmask]

    raw_maps[cond] = tf_raw_roi
    db_maps[cond]  = tf_db_roi
    roi_times = times_roi

    # сохранить 4 raw
    save_tfr_heatmap(
        tf_raw_roi, times_roi, freqs,
        title=f"{sub} | {cond} | TF RAW (avg grad & trials) | ROI {roi[0]}..{roi[1]} s",
        fname=f"{sub}_{cond}_TF_RAW_ROI.png"
    )

    # сохранить 4 baseline-corrected dB
    save_tfr_heatmap(
        tf_db_roi, times_roi, freqs,
        title=f"{sub} | {cond} | TF baseline-corrected (dB) | ROI {roi[0]}..{roi[1]} s",
        fname=f"{sub}_{cond}_TF_dB_ROI.png"
    )

print("Saved 8 plots: 4 RAW + 4 dB.")

Reading C:\Users\Yulia\Desktop\Master thesis\sub-001\sub-001_sivi-epo.fif ...
    Found the data of interest:
        t =   -4000.00 ...    3500.00 ms
        0 CTF compensation matrices available
Not setting metadata
54 matching events found
No baseline correction applied
0 projection items activated
Applying baseline correction (mode: logratio)
Reading C:\Users\Yulia\Desktop\Master thesis\sub-001\sub-001_sive-epo.fif ...
    Found the data of interest:
        t =   -4000.00 ...    3500.00 ms
        0 CTF compensation matrices available
Not setting metadata
59 matching events found
No baseline correction applied
0 projection items activated
Applying baseline correction (mode: logratio)
Reading C:\Users\Yulia\Desktop\Master thesis\sub-001\sub-001_covi-epo.fif ...
    Found the data of interest:
        t =   -4000.00 ...    3500.00 ms
        0 CTF compensation matrices available
Not setting metadata
55 matching events found
No baseline correction applied
0 projection items activated

In [19]:
diff1 = db_maps["covi"] - db_maps["sivi"]
diff2 = db_maps["cove"] - db_maps["sive"]

save_tfr_heatmap(
    diff1, roi_times, freqs,
    title=f"{sub} | covi - sivi | TF diff (dB) | ROI {roi[0]}..{roi[1]} s",
    fname=f"{sub}_DIFF_covi_minus_sivi_dB_ROI.png"
)

save_tfr_heatmap(
    diff2, roi_times, freqs,
    title=f"{sub} | cove - sive | TF diff (dB) | ROI {roi[0]}..{roi[1]} s",
    fname=f"{sub}_DIFF_cove_minus_sive_dB_ROI.png"
)

print("Saved 2 diff plots. Total = 10 plots in:", out_dir)

Saved 2 diff plots. Total = 10 plots in: sub-001\TFR_10plots
